In [1]:
# Step 1: PyTorch 다운그레이드
!pip install torch==2.1.0 torchvision==0.16.0 \
    --index-url https://download.pytorch.org/whl/cu118 -q

# Step 2: NumPy 다운그레이드
!pip install "numpy<2" -q

print("=== 다운그레이드 완료. 세션 재시작 후 다음 셀로 ===")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 162.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 33.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.1.0+cu118 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 147.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requir

In [2]:
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")

Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
PyTorch: 2.1.0+cu118


In [ ]:
import os
from google.colab import userdata

# Secrets에서 토큰 가져오기
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Kaggle CLI 설치
!pip install -q kaggle

# 본인 Dataset 다운로드 (wony98/pill-detection-preprocessed v2)
%cd /content
!kaggle datasets download -d wony98/pill-detection-preprocessed --unzip -p /content/dataset_yolo

# 검증
import glob
print(f"\n=== 다운로드 검증 ===")
print(f"train images: {len(glob.glob('/content/dataset_yolo/images/train/*'))}")
print(f"val images:   {len(glob.glob('/content/dataset_yolo/images/val/*'))}")
print(f"pill.yaml exists: {os.path.exists('/content/dataset_yolo/pill.yaml')}")

/content
Dataset URL: https://www.kaggle.com/datasets/wony98/pill-detection-preprocessed
License(s): CC0-1.0
100% 17.5G/17.5G [00:47<00:00, 338MB/s]
100% 17.5G/17.5G [00:47<00:00, 396MB/s]

=== 다운로드 검증 ===
train images: 8302
val images:   2090
pill.yaml exists: True


In [ ]:
import os, glob

# 업로드 위치 확인
print("=== /content/ 파일 목록 ===")
!ls -lh /content/*.zip 2>/dev/null

# synthetic.zip 압축 해제
if os.path.exists('/content/synthetic.zip'):
    !rm -rf /content/synthetic
    !unzip -q /content/synthetic.zip -d /content/synthetic

    print(f"\n=== 압축 해제 검증 ===")
    print(f"images/train: {len(glob.glob('/content/synthetic/images/train/*'))}")
    print(f"images/val:   {len(glob.glob('/content/synthetic/images/val/*'))}")
    print(f"labels/train: {len(glob.glob('/content/synthetic/labels/train/*'))}")
    print(f"labels/val:   {len(glob.glob('/content/synthetic/labels/val/*'))}")
else:
    print("\n⚠️ /content/synthetic.zip 없음 — 업로드 위치 확인 필요")

=== /content/ 파일 목록 ===
-rw-r--r-- 1 root root  18M Apr 29 05:45 /content/eval_package.zip
-rw-r--r-- 1 root root 801M Apr 29 05:52 /content/synthetic.zip
-rw-r--r-- 1 root root  11K Apr 29 05:46 /content/test_labels.zip

=== 압축 해제 검증 ===
images/train: 1800
images/val:   200
labels/train: 1800
labels/val:   200


In [1]:
# mmdet 재설치 (세션 재시작했으니 다시)
!pip install -U openmim -q
!mim install mmengine -q
!mim install "mmcv>=2.0.0,<2.2.0" -q

# mmdetection 소스 (이미 clone된 게 있으면 재사용)
import os
if not os.path.exists('/content/mmdetection'):
    !git clone https://github.com/open-mmlab/mmdetection.git /content/mmdetection

%cd /content/mmdetection
!pip install -v -e . -q

print("\n=== 설치 완료, import 검증 ===")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 24.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.4/239.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 13.9 MB/s eta 0:0

In [ ]:
import mmengine, mmcv, mmdet
from mmcv.ops import RoIAlign
from mmdet.registry import MODELS

print(f"mmengine : {mmengine.__version__}")
print(f"mmcv     : {mmcv.__version__}")
print(f"mmdet    : {mmdet.__version__}")
print(f"\n✅ 모두 정상")

mmengine : 0.10.7
mmcv     : 2.1.0
mmdet    : 3.3.0

✅ 모두 정상


In [ ]:
import yaml

# pill.yaml 로드
with open('/content/dataset_yolo/pill.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print(f"=== pill.yaml ===")
print(f"클래스 수: {len(cfg.get('names', []))}")
print(f"첫 5개 클래스: {cfg.get('names', [])[:5]}")
print(f"마지막 5개:   {cfg.get('names', [])[-5:]}")

# 합성 데이터 class_id 확인
import glob
synth_classes = set()
for txt in glob.glob('/content/synthetic/labels/train/*.txt')[:100]:
    with open(txt) as f:
        for line in f:
            if line.strip():
                synth_classes.add(int(line.split()[0]))

print(f"\n=== synthetic train labels (첫 100파일 샘플) ===")
print(f"발견된 class_id 수: {len(synth_classes)}")
print(f"min/max: {min(synth_classes)}/{max(synth_classes)}")

=== pill.yaml ===
클래스 수: 118
첫 5개 클래스: ['마그밀정(수산화마그네슘)', '게보린정 300mg/PTP', '알마겔정(알마게이트)(수출명:유한가스트라겔정)', '보령부스파정 5mg', '뮤테란캡슐 100mg']
마지막 5개:   ['바이오탑디캡슐 50mg/병', '케이캡정 50mg', '바이오탑하이캡슐 150mg/병', '엘스테인정(에르도스테인)', '디카테오엘씨정 10mg/병']

=== synthetic train labels (첫 100파일 샘플) ===
발견된 class_id 수: 105
min/max: 1/117


In [ ]:
import os, json, glob
from PIL import Image
from tqdm import tqdm
import yaml

# 클래스 이름 로드
with open('/content/dataset_yolo/pill.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']  # 118개

# COCO categories (1-based, COCO 표준)
categories = [{"id": i+1, "name": name, "supercategory": "pill"}
              for i, name in enumerate(class_names)]

def yolo_to_coco(image_dirs, label_dirs, output_path):
    """YOLO 포맷 → COCO JSON. 여러 디렉터리 통합 가능."""
    images, annotations = [], []
    img_id, ann_id = 1, 1

    for img_dir, lbl_dir in zip(image_dirs, label_dirs):
        img_files = sorted(glob.glob(f'{img_dir}/*'))
        for img_path in tqdm(img_files, desc=os.path.basename(img_dir)):
            fname = os.path.basename(img_path)
            stem = os.path.splitext(fname)[0]
            lbl_path = f'{lbl_dir}/{stem}.txt'

            # 이미지 크기
            try:
                with Image.open(img_path) as im:
                    W, H = im.size
            except Exception as e:
                print(f"이미지 로드 실패: {img_path} - {e}")
                continue

            # 이미지 등록 (mmdet은 file_name에 상대 경로 또는 파일명)
            images.append({
                "id": img_id,
                "file_name": fname,
                "width": W,
                "height": H,
                "abs_path": img_path,  # mmdet 학습용 (커스텀 파이프라인에서 활용)
            })

            # 라벨 등록
            if os.path.exists(lbl_path):
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) != 5:
                            continue
                        cls, cx, cy, w, h = map(float, parts)
                        # YOLO normalized → COCO [x,y,w,h] pixel
                        bw, bh = w * W, h * H
                        bx = cx * W - bw / 2
                        by = cy * H - bh / 2
                        annotations.append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": int(cls) + 1,  # 0-based → 1-based
                            "bbox": [bx, by, bw, bh],
                            "area": bw * bh,
                            "iscrowd": 0,
                        })
                        ann_id += 1
            img_id += 1

    coco = {"images": images, "annotations": annotations, "categories": categories}
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f"\n✅ {output_path}")
    print(f"   images: {len(images)}, annotations: {len(annotations)}")

# Train: dataset_yolo train + synthetic train 통합
yolo_to_coco(
    image_dirs=['/content/dataset_yolo/images/train', '/content/synthetic/images/train'],
    label_dirs=['/content/dataset_yolo/labels/train', '/content/synthetic/labels/train'],
    output_path='/content/train_coco.json'
)

# Val: dataset_yolo val + synthetic val 통합
yolo_to_coco(
    image_dirs=['/content/dataset_yolo/images/val', '/content/synthetic/images/val'],
    label_dirs=['/content/dataset_yolo/labels/val', '/content/synthetic/labels/val'],
    output_path='/content/val_coco.json'
)

train: 100%|██████████| 1800/1800 [00:00<00:00, 8908.57it/s]



✅ /content/train_coco.json
   images: 10102, annotations: 36209


val: 100%|██████████| 200/200 [00:00<00:00, 8673.35it/s]



✅ /content/val_coco.json
   images: 2290, annotations: 8515


In [ ]:
import json

for path in ['/content/train_coco.json', '/content/val_coco.json']:
    with open(path) as f:
        coco = json.load(f)

    # 중복 file_name 확인 (basename 기준)
    fnames = [img['file_name'] for img in coco['images']]
    dup_count = len(fnames) - len(set(fnames))
    print(f"\n{path}")
    print(f"  중복 basename: {dup_count}")

    # file_name = abs_path로 교체
    for img in coco['images']:
        img['file_name'] = img.pop('abs_path')

    # 절대경로 중복 재확인
    abs_names = [img['file_name'] for img in coco['images']]
    print(f"  절대경로 중복: {len(abs_names) - len(set(abs_names))}")

    # 저장
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f"  ✅ 저장 완료")

# 첫 이미지 샘플 확인
import json
with open('/content/train_coco.json') as f:
    sample = json.load(f)
print(f"\n=== 샘플 ===")
print(f"첫 이미지: {sample['images'][0]['file_name']}")
print(f"마지막 이미지: {sample['images'][-1]['file_name']}")


/content/train_coco.json
  중복 basename: 0
  절대경로 중복: 0
  ✅ 저장 완료

/content/val_coco.json
  중복 basename: 0
  절대경로 중복: 0
  ✅ 저장 완료

=== 샘플 ===
첫 이미지: /content/dataset_yolo/images/train/K-000250-000573-002483-012778_0_2_0_2_75_000_200.png
마지막 이미지: /content/synthetic/images/train/synth_01799.jpg


In [ ]:
import os

# 작업 디렉터리
os.makedirs('/content/rtmdet_work', exist_ok=True)
os.chdir('/content/rtmdet_work')

# 클래스 이름 로드 (118개)
import yaml
with open('/content/dataset_yolo/pill.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
class_names = tuple(cfg['names'])

# Custom config 작성
config_text = f"""
# RTMDet-tiny custom config (Pill Detection 118 classes)
# Base: rtmdet_tiny_8xb32-300e_coco.py
_base_ = '/content/mmdetection/configs/rtmdet/rtmdet_tiny_8xb32-300e_coco.py'

# === 데이터 ===
data_root = ''  # file_name이 절대경로라 빈 값
metainfo = {{
    'classes': {class_names!r},
}}

# === 모델: num_classes 변경 ===
model = dict(
    bbox_head=dict(num_classes=118),
)

# === 학습 셋팅 (팀 통일 영역) ===
max_epochs = 15
train_cfg = dict(
    max_epochs=max_epochs,
    val_interval=5,  # 5 epoch마다 val
)

# === Dataloader ===
train_dataloader = dict(
    batch_size=16,
    num_workers=4,
    dataset=dict(
        data_root=data_root,
        ann_file='/content/train_coco.json',
        data_prefix=dict(img=''),  # file_name이 절대경로라 prefix 빈 값
        metainfo=metainfo,
    ),
)
val_dataloader = dict(
    batch_size=16,
    num_workers=4,
    dataset=dict(
        data_root=data_root,
        ann_file='/content/val_coco.json',
        data_prefix=dict(img=''),
        metainfo=metainfo,
    ),
)
test_dataloader = val_dataloader

# === Evaluator ===
val_evaluator = dict(ann_file='/content/val_coco.json')
test_evaluator = val_evaluator

# === Optimizer / LR ===
# 기본 lr=0.004는 batch=32 8GPU 기준. 우리 batch=16 1GPU라 비례 축소
base_lr = 0.004 / 16  # = 0.00025
optim_wrapper = dict(
    optimizer=dict(lr=base_lr),
)

# === Random Seed ===
randomness = dict(seed=100)

# === 사전학습 weights ===
load_from = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth'

# === 체크포인트 저장 (best mAP 기준) ===
default_hooks = dict(
    checkpoint=dict(interval=5, max_keep_ckpts=2, save_best='coco/bbox_mAP'),
)

# === Work dir ===
work_dir = '/content/rtmdet_work_seed100'
"""

config_path = '/content/rtmdet_work/rtmdet_tiny_pill.py'
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_text)

print(f"✅ Config 저장: {config_path}")
print(f"\n=== Config 내용 (요약) ===")
print(f"  num_classes  : 118")
print(f"  epochs       : 15")
print(f"  batch        : 16")
print(f"  imgsz        : 640 (RTMDet 표준)")
print(f"  seed         : 100")
print(f"  lr (조정)    : {0.004 / 16:.5f}")
print(f"  pretrained   : RTMDet-tiny COCO weights")

✅ Config 저장: /content/rtmdet_work/rtmdet_tiny_pill.py

=== Config 내용 (요약) ===
  num_classes  : 118
  epochs       : 15
  batch        : 16
  imgsz        : 640 (RTMDet 표준)
  seed         : 100
  lr (조정)    : 0.00025
  pretrained   : RTMDet-tiny COCO weights


In [ ]:
from mmengine.config import Config
from mmengine.registry import init_default_scope

# Config 로드
cfg = Config.fromfile('/content/rtmdet_work/rtmdet_tiny_pill.py')

# scope 초기화 (mmdet 모듈 등록)
init_default_scope('mmdet')

# 핵심 설정 출력
print("=== Config 파싱 결과 ===")
print(f"max_epochs       : {cfg.train_cfg.max_epochs}")
print(f"train batch_size : {cfg.train_dataloader.batch_size}")
print(f"num_classes      : {cfg.model.bbox_head.num_classes}")
print(f"lr               : {cfg.optim_wrapper.optimizer.lr}")
print(f"seed             : {cfg.randomness.seed}")
print(f"train ann_file   : {cfg.train_dataloader.dataset.ann_file}")
print(f"val ann_file     : {cfg.val_dataloader.dataset.ann_file}")
print(f"work_dir         : {cfg.work_dir}")

# 모델 빌드 시도 (학습 시작 전 검증)
print("\n=== 모델 빌드 시도 ===")
from mmdet.registry import MODELS
model = MODELS.build(cfg.model)
print(f"✅ 모델 빌드 성공")
print(f"   파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# Dataset 빌드 시도
print("\n=== Train Dataset 빌드 시도 ===")
from mmdet.registry import DATASETS
train_ds = DATASETS.build(cfg.train_dataloader.dataset)
print(f"✅ Train dataset 빌드 성공")
print(f"   샘플 수: {len(train_ds)}")
print(f"   첫 샘플 file_name: {train_ds[0]['data_samples'].img_path if hasattr(train_ds[0]['data_samples'], 'img_path') else train_ds[0]}")

=== Config 파싱 결과 ===
max_epochs       : 15
train batch_size : 16
num_classes      : 118
lr               : 0.00025
seed             : 100
train ann_file   : /content/train_coco.json
val ann_file     : /content/val_coco.json
work_dir         : /content/rtmdet_work_seed100

=== 모델 빌드 시도 ===
✅ 모델 빌드 성공
   파라미터 수: 4.91M

=== Train Dataset 빌드 시도 ===
loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
✅ Train dataset 빌드 성공
   샘플 수: 10102
   첫 샘플 file_name: /content/dataset_yolo/images/train/K-000250-000573-002483-012778_0_2_0_2_75_000_200.png


In [ ]:
import json, os

for path in ['/content/train_coco.json', '/content/val_coco.json']:
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    patched = False
    if 'info' not in data:
        data['info'] = {}
        patched = True
    if 'licenses' not in data:
        data['licenses'] = []
        patched = True
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"{os.path.basename(path)}: info={'info' in data}, licenses={'licenses' in data}, patched={patched}, size={os.path.getsize(path)//1024}KB")

train_coco.json: info=True, licenses=True, patched=True, size=6951KB
val_coco.json: info=True, licenses=True, patched=True, size=1624KB


In [ ]:
%cd /content/rtmdet_work

# 학습 명령
!python /content/mmdetection/tools/train.py \
    /content/rtmdet_work/rtmdet_tiny_pill.py \
    --work-dir /content/rtmdet_work

/content/mmdetection
04/29 06:02:34 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 100
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.5, V12.5.82
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gen

In [ ]:
!python /content/mmdetection/tools/train.py \
    /content/rtmdet_work/rtmdet_tiny_pill.py \
    --work-dir /content/rtmdet_work \
    --resume

In [ ]:
!ls -la /content/rtmdet_work/

In [ ]:
import os, zipfile

# 평가셋 압축 해제
if os.path.exists('/content/eval_package.zip'):
    !rm -rf /content/eval_package
    with zipfile.ZipFile('/content/eval_package.zip', 'r') as z:
        z.extractall('/content/eval_package')

    import glob
    print(f"평가 이미지: {len(glob.glob('/content/eval_package/test_images/*'))}")
    print(f"answer_key 존재: {os.path.exists('/content/eval_package/answer_key.json')}")
else:
    print("⚠️ /content/eval_package.zip 업로드 필요")

평가 이미지: 33
answer_key 존재: True


In [ ]:
import glob, os

# best.pt 찾기 (RTMDet은 best_coco_bbox_mAP_epoch_X.pth 형태로 저장)
best_paths = sorted(glob.glob('/content/rtmdet_work/best_coco_bbox_mAP_*.pth'))
if not best_paths:
    # 못 찾으면 last epoch checkpoint 사용
    best_paths = sorted(glob.glob('/content/rtmdet_work/epoch_*.pth'))

assert best_paths, "체크포인트 못 찾음"
BEST_PT = best_paths[-1]
CONFIG = '/content/rtmdet_work/rtmdet_tiny_pill.py'
print(f"✅ Best checkpoint: {BEST_PT}")
print(f"✅ Config: {CONFIG}")

# 모델 로드
from mmdet.apis import init_detector
model = init_detector(CONFIG, BEST_PT, device='cuda:0')
print(f"\n✅ 모델 로드 완료")

✅ Best checkpoint: /content/rtmdet_work/best_coco_bbox_mAP_epoch_15.pth
✅ Config: /content/rtmdet_work/rtmdet_tiny_pill.py
Loads checkpoint by local backend from path: /content/rtmdet_work/best_coco_bbox_mAP_epoch_15.pth

✅ 모델 로드 완료


In [ ]:
import numpy as np
import unicodedata

def nfc(s):
    return unicodedata.normalize('NFC', s) if s else s

def iou_xyxy(b1, b2):
    """[x1,y1,x2,y2] IoU"""
    xa, ya = max(b1[0], b2[0]), max(b1[1], b2[1])
    xb, yb = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, xb-xa) * max(0, yb-ya)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0

def xyxy_to_xywh(b):
    """[x1,y1,x2,y2] → [x,y,w,h] (COCO 포맷)"""
    return [b[0], b[1], b[2]-b[0], b[3]-b[1]]

def extract_top3_groups(bboxes, scores, labels, class_names, iou_thr=0.5):
    """
    낮은 conf로 받은 raw 박스들을 IoU>0.5로 그룹핑해서 같은 위치의 Top-3 라벨 뽑기.
    bboxes: (N, 4) xyxy / scores: (N,) / labels: (N,)
    """
    if len(bboxes) == 0:
        return []

    # conf 내림차순 정렬
    order = np.argsort(-scores)
    bboxes, scores, labels = bboxes[order], scores[order], labels[order]

    groups = []  # [{rep_bbox, candidates: [(score, label), ...]}, ...]
    used = np.zeros(len(bboxes), dtype=bool)

    for i in range(len(bboxes)):
        if used[i]:
            continue
        rep_bbox = bboxes[i]
        candidates = [(scores[i], labels[i])]
        used[i] = True
        # 같은 그룹 박스 수집
        for j in range(i+1, len(bboxes)):
            if used[j]:
                continue
            if iou_xyxy(rep_bbox, bboxes[j]) > iou_thr:
                candidates.append((scores[j], labels[j]))
                used[j] = True
        groups.append({'bbox': rep_bbox, 'candidates': candidates})

    # 각 그룹에서 class별 best conf만 남기고 Top-3
    result = []
    for g in groups:
        cls_best = {}
        for s, l in g['candidates']:
            if l not in cls_best or s > cls_best[l]:
                cls_best[l] = s
        sorted_cls = sorted(cls_best.items(), key=lambda x: -x[1])[:3]
        top3_labels = [int(l) for _, l in sorted_cls]
        top3_scores = [float(s) for s, _ in sorted_cls]
        top3_names = [class_names[l] for l in top3_labels]
        result.append({
            'bbox': xyxy_to_xywh(g['bbox'].tolist()),
            'top3_class_idx': top3_labels,
            'top3_class_name': top3_names,
            'top3_confidence': top3_scores,
        })
    return result

print("✅ 헬퍼 함수 정의 완료")

✅ 헬퍼 함수 정의 완료


In [ ]:
import json, glob, time, yaml
from mmdet.apis import inference_detector

# 클래스 이름 (COCO category_id는 1-based지만 모델 출력은 0-based label)
with open('/content/dataset_yolo/pill.yaml', 'r', encoding='utf-8') as f:
    class_names = yaml.safe_load(f)['names']

# answer_key (GT) 로드
with open('/content/eval_package/answer_key.json', 'r', encoding='utf-8') as f:
    answer_key = json.load(f)

# 33장 inference
test_imgs = sorted(glob.glob('/content/eval_package/test_images/*'))
print(f"평가 이미지 수: {len(test_imgs)}")

# Inference 셋팅 (RTMDet conf 0.01로 충분한 후보 확보, NMS iou 0.6)
CONF_THR = 0.01
NMS_IOU = 0.6  # config 기본값. RTMDet inference 시 model.test_cfg에서 NMS 자동 적용

results = []
t_start = time.time()
for img_path in test_imgs:
    fname = os.path.basename(img_path)

    # 모델 inference
    pred = inference_detector(model, img_path)

    # 결과 추출 (mmdet 3.x: pred.pred_instances)
    inst = pred.pred_instances
    bboxes = inst.bboxes.cpu().numpy()  # xyxy
    scores = inst.scores.cpu().numpy()
    labels = inst.labels.cpu().numpy()

    # conf threshold 필터
    mask = scores >= CONF_THR
    bboxes, scores, labels = bboxes[mask], scores[mask], labels[mask]

    # Top-3 그룹핑
    predictions = extract_top3_groups(bboxes, scores, labels, class_names)

    # GT (answer_key 형식: {filename: {dl_name, class_idx, ...}})
    gt_info = answer_key.get(fname, None)
    if gt_info is None:
        # 다중 알약 케이스 가능성 — answer_key에 없으면 빈 리스트
        gt_bboxes = []
    else:
        # answer_key는 이미지당 GT bbox 정보가 있어야 함 (포맷 확인 필요)
        # 메모리상 answer_key는 이미지당 dl_name만 — bbox는 별도 GT 필요
        # 일단 answer_key의 dl_name만 기록, bbox는 채점 단계에서 별도 처리
        gt_bboxes = [{'dl_name': nfc(gt_info['dl_name'])}]

    results.append({
        'filename': fname,
        'gt_bboxes': gt_bboxes,
        'predictions': predictions,
    })

ms_per_img = (time.time() - t_start) * 1000 / len(test_imgs)

# v4 JSON 형식 (귀한님과 동일)
v4_json = {
    'exp_name': 'rtmdet_tiny_synth_e15_seed100',
    'author': '승원',
    'model_framework': 'mmdet',
    'model_params_M': 4.91,
    'batch_size_used': 16,
    'seed': 100,
    'conf_threshold': CONF_THR,
    'nms_iou': NMS_IOU,
    'augmentations': ['RTMDet default (CachedMosaic + RandomResize + RandomCrop + YOLOXHSVRandomAug + RandomFlip)'],
    'ms_per_image': round(ms_per_img, 2),
    'num_images': len(test_imgs),
    'num_gt_bboxes': sum(len(r['gt_bboxes']) for r in results),
    'results': results,
}

# 저장
output_path = '/content/rtmdet_tiny_synth_e15_seed100_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(v4_json, f, ensure_ascii=False, indent=2)

print(f"\n✅ {output_path}")
print(f"   ms/img: {ms_per_img:.2f}")
print(f"   num_gt_bboxes: {v4_json['num_gt_bboxes']}")

평가 이미지 수: 33


/usr/local/lib/python3.11/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



✅ /content/rtmdet_tiny_synth_e15_seed100_results.json
   ms/img: 157.84
   num_gt_bboxes: 33


In [ ]:
import shutil

# best.pt + config + 결과 JSON 묶어서 zip
import os, glob

# 학습 산출물 핵심만 추리기
backup_dir = '/content/rtmdet_backup'
os.makedirs(backup_dir, exist_ok=True)

# best.pt 복사
best_paths = sorted(glob.glob('/content/rtmdet_work/best_coco_bbox_mAP_*.pth'))
if not best_paths:
    best_paths = sorted(glob.glob('/content/rtmdet_work/epoch_*.pth'))
shutil.copy(best_paths[-1], backup_dir)

# config 복사
shutil.copy('/content/rtmdet_work/rtmdet_tiny_pill.py', backup_dir)

# v4 결과 JSON 복사
shutil.copy('/content/rtmdet_tiny_synth_e15_seed100_results.json', backup_dir)

# 학습 로그 복사 (선택)
log_dirs = sorted(glob.glob('/content/rtmdet_work/2*'))  # timestamp 폴더
if log_dirs:
    shutil.copytree(log_dirs[-1], f'{backup_dir}/training_log', dirs_exist_ok=True)

# zip 생성
shutil.make_archive('/content/rtmdet_backup', 'zip', backup_dir)
size_mb = os.path.getsize('/content/rtmdet_backup.zip') / 1e6
print(f"✅ /content/rtmdet_backup.zip ({size_mb:.1f} MB)")
print(f"   좌측 파일 탭에서 우클릭 → 다운로드")

✅ /content/rtmdet_backup.zip (39.5 MB)
   좌측 파일 탭에서 우클릭 → 다운로드


In [ ]:
import os, glob, zipfile

# zip이면 풀기
if os.path.exists('/content/test_labels.zip'):
    with zipfile.ZipFile('/content/test_labels.zip', 'r') as z:
        z.extractall('/content/')
    print("test_labels.zip 압축 해제 완료")

# 라벨 폴더 찾기
candidates = []
for path in glob.glob('/content/**/*.txt', recursive=True):
    if 'test_labels' in path or 'eval_package' not in path:
        candidates.append(path)

# test_labels 폴더 추정
if os.path.isdir('/content/test_labels'):
    label_dir = '/content/test_labels'
elif os.path.isdir('/content/test_labels/test_labels'):
    label_dir = '/content/test_labels/test_labels'
else:
    label_dir = None

if label_dir:
    txts = glob.glob(f'{label_dir}/*.txt')
    print(f"\n라벨 폴더: {label_dir}")
    print(f".txt 파일 수: {len(txts)} (33이어야 정상)")

    # 첫 파일 내용 확인
    if txts:
        with open(txts[0], 'r') as f:
            content = f.read()
        print(f"\n샘플 ({os.path.basename(txts[0])}):")
        print(content)

    # 47 bbox 합계 검증
    total = 0
    for txt in txts:
        with open(txt, 'r') as f:
            total += sum(1 for line in f if line.strip())
    print(f"\n총 bbox 수: {total} (47이어야 정상)")
else:
    print("test_labels 폴더 못 찾음. 업로드 위치 확인 필요.")

test_labels.zip 압축 해제 완료

라벨 폴더: /content/test_labels
.txt 파일 수: 33 (33이어야 정상)

샘플 (╕╢└╠─«╡≡┴ñ.txt):
72 0.504654 0.261307 0.411582 0.182580
72 0.480352 0.476549 0.427094 0.167504


총 bbox 수: 47 (47이어야 정상)


In [ ]:
import os, glob

# 1. test_images 파일명 첫 5개
print("=== test_images (정상 한글) ===")
imgs = sorted(glob.glob('/content/eval_package/test_images/*'))
for p in imgs[:5]:
    print(f"  {os.path.basename(p)}")

# 2. test_labels 파일명 첫 5개
print("\n=== test_labels (깨졌는지 확인) ===")
labels = sorted(glob.glob('/content/test_labels/*.txt'))
for p in labels[:5]:
    print(f"  {os.path.basename(p)}")

# 3. 매칭 검증 (확장자 빼고 stem 비교)
img_stems = {os.path.splitext(os.path.basename(p))[0] for p in imgs}
label_stems = {os.path.splitext(os.path.basename(p))[0] for p in labels}
matched = img_stems & label_stems
print(f"\n=== 매칭 ===")
print(f"이미지 stem: {len(img_stems)}, 라벨 stem: {len(label_stems)}")
print(f"매칭됨: {len(matched)} / 33")
if len(matched) < 33:
    print(f"이미지에만: {sorted(img_stems - label_stems)[:3]}")
    print(f"라벨에만: {sorted(label_stems - img_stems)[:3]}")

=== test_images (정상 한글) ===
  게보린정 300mg_3.png
  글리아타민연질캡슐.png
  낙소졸정 500_20mg.png
  넥시움정 40mg.png
  노바스크정 5mg.png

=== test_labels (깨졌는지 확인) ===
  ─φ┐í┼╕╟╔┴ñ 25mg.txt
  ─╔└╠─╕┴ñ 50mg.txt
  │½╝╥┴╣┴ñ 500_20mg.txt
  │δ╣┘╜║┼⌐┴ñ 5mg.txt
  │ε┼╪┴ñ 10mg.txt

=== 매칭 ===
이미지 stem: 33, 라벨 stem: 33
매칭됨: 0 / 33
이미지에만: ['게보린정 300mg_3', '글리아타민연질캡슐', '낙소졸정 500_20mg']
라벨에만: ['─φ┐í┼╕╟╔┴ñ 25mg', '─╔└╠─╕┴ñ 50mg', '│½╝╥┴╣┴ñ 500_20mg']


In [ ]:
import zipfile, os, shutil, glob

# 기존 풀린 폴더 삭제
shutil.rmtree('/content/test_labels', ignore_errors=True)
os.makedirs('/content/test_labels', exist_ok=True)

# 인코딩 후보 순서대로 시도
ENCODING_CANDIDATES = ['cp949', 'euc-kr', 'utf-8']

def fix_name(raw):
    """cp437로 잘못 디코딩된 이름을 한글로 복원"""
    try:
        raw_bytes = raw.encode('cp437')
    except UnicodeEncodeError:
        return raw
    for enc in ENCODING_CANDIDATES:
        try:
            return raw_bytes.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw

with zipfile.ZipFile('/content/test_labels.zip', 'r') as z:
    for info in z.infolist():
        fixed = fix_name(info.filename)
        if info.is_dir():
            os.makedirs(os.path.join('/content/test_labels', fixed), exist_ok=True)
            continue
        target = os.path.join('/content/test_labels', fixed)
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)

# 검증
labels = sorted(glob.glob('/content/test_labels/**/*.txt', recursive=True))
print(f".txt 파일 수: {len(labels)}")
print("\n첫 5개 파일명:")
for p in labels[:5]:
    print(f"  {os.path.basename(p)}")

# 매칭 재검증
imgs = sorted(glob.glob('/content/eval_package/test_images/*'))
img_stems = {os.path.splitext(os.path.basename(p))[0] for p in imgs}
label_stems = {os.path.splitext(os.path.basename(p))[0] for p in labels}
matched = img_stems & label_stems
print(f"\n매칭: {len(matched)} / 33")
if len(matched) < 33:
    print(f"이미지에만: {sorted(img_stems - label_stems)[:3]}")
    print(f"라벨에만: {sorted(label_stems - img_stems)[:3]}")

# bbox 합계
total = 0
for txt in labels:
    with open(txt, 'r') as f:
        total += sum(1 for line in f if line.strip())
print(f"총 bbox 수: {total} (47이어야 정상)")

.txt 파일 수: 33

첫 5개 파일명:
  게보린정 300mg_3.txt
  글리아타민연질캡슐.txt
  낙소졸정 500_20mg.txt
  넥시움정 40mg.txt
  노바스크정 5mg.txt

매칭: 33 / 33
총 bbox 수: 47 (47이어야 정상)


In [ ]:
import csv

CSV_PATH = '/content/pill_names_118.csv'

with open(CSV_PATH, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    rows = list(reader)

print(f"총 행 수: {len(rows)}")
print(f"\n=== 첫 5행 ===")
for i, row in enumerate(rows[:5]):
    print(f"  [{i}] {row}")
print(f"\n=== 마지막 2행 ===")
for row in rows[-2:]:
    print(f"  {row}")

In [ ]:
import yaml, unicodedata, os, glob
from PIL import Image

# 매핑 (이전 셀에서 검증됨, 다시)
with open('/content/dataset_yolo/pill.yaml', 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)
class_idx_to_name = {i: unicodedata.normalize('NFC', n) for i, n in enumerate(yaml_data['names'])}

# 경로 자동 탐지 (중첩이든 평탄이든 둘 다 대응)
LABEL_DIR_CANDIDATES = ['/content/test_labels/test_labels', '/content/test_labels']
LABEL_DIR = None
for cand in LABEL_DIR_CANDIDATES:
    if os.path.isdir(cand) and glob.glob(f'{cand}/*.txt'):
        LABEL_DIR = cand
        break

IMG_DIR = '/content/eval_package/test_images'
print(f"라벨 폴더: {LABEL_DIR}")
print(f"라벨 .txt 수: {len(glob.glob(f'{LABEL_DIR}/*.txt'))}")

# img_files dict
img_files = {}
for p in glob.glob(f'{IMG_DIR}/*'):
    stem = unicodedata.normalize('NFC', os.path.splitext(os.path.basename(p))[0])
    img_files[stem] = p

# GT 빌드
GT_LABELS = {}
unmatched = []
for txt_path in sorted(glob.glob(f'{LABEL_DIR}/*.txt')):
    stem = unicodedata.normalize('NFC', os.path.splitext(os.path.basename(txt_path))[0])
    if stem not in img_files:
        unmatched.append(stem)
        continue
    img_w, img_h = Image.open(img_files[stem]).size
    bboxes = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            cx, cy, w, h = map(float, parts[1:])
            bboxes.append({
                'bbox': [(cx-w/2)*img_w, (cy-h/2)*img_h, (cx+w/2)*img_w, (cy+h/2)*img_h],
                'class_idx': cls,
                'dl_name': class_idx_to_name.get(cls, f'unknown_{cls}'),
            })
    GT_LABELS[stem] = {'img_path': img_files[stem], 'img_width': img_w, 'img_height': img_h, 'bboxes': bboxes}

total_gt = sum(len(v['bboxes']) for v in GT_LABELS.values())
print(f"\nGT 이미지: {len(GT_LABELS)}장 / 33")
print(f"GT bbox 합계: {total_gt} / 47")
if unmatched:
    print(f"\n⚠️ 매칭 실패 라벨 ({len(unmatched)}개): {unmatched[:3]}")

multi = [(s, [b['dl_name'] for b in v['bboxes']]) for s, v in GT_LABELS.items() if len(v['bboxes']) > 1]
print(f"\n다중 알약 이미지 ({len(multi)}장):")
for s, names in multi[:5]:
    print(f"  {s}: {names}")

라벨 폴더: /content/test_labels/test_labels
라벨 .txt 수: 33

GT 이미지: 33장 / 33
GT bbox 합계: 47 / 47

다중 알약 이미지 (13장):
  놀텍정 10mg: ['놀텍정 10mg', '놀텍정 10mg']
  디카테오정: ['디카테오정', '디카테오정', '디카테오정']
  라시도필캡슐(비매품): ['라시도필캡슐(비매품)', '라시도필캡슐(비매품)']
  마이칼디정: ['마이칼디정', '마이칼디정']
  맥시부펜이알정: ['맥시부펜이알정 300mg', '맥시부펜이알정 300mg']


In [ ]:
from mmdet.apis import init_detector, inference_detector
import os

CONFIG = '/content/rtmdet_work/rtmdet_tiny_pill.py'
BEST_PT = '/content/rtmdet_work/best_coco_bbox_mAP_epoch_15.pth'

# 모델 로드
model = init_detector(CONFIG, BEST_PT, device='cuda:0')
print(f"모델 로드 OK")
print(f"num_classes: {model.dataset_meta.get('classes', 'N/A')[:3] if model.dataset_meta else 'N/A'}... (총 {len(model.dataset_meta['classes']) if model.dataset_meta else 0})")

# 단일 이미지 inference
sample_stem = list(GT_LABELS.keys())[0]
sample_path = GT_LABELS[sample_stem]['img_path']
print(f"\n샘플: {sample_stem}")
print(f"GT bbox 수: {len(GT_LABELS[sample_stem]['bboxes'])}")
print(f"GT class_idx: {[b['class_idx'] for b in GT_LABELS[sample_stem]['bboxes']]}")

result = inference_detector(model, sample_path)

# DetDataSample 구조 확인
print(f"\n=== inference 결과 타입 ===")
print(f"  type: {type(result).__name__}")
print(f"  pred_instances 키: {list(result.pred_instances.keys()) if hasattr(result, 'pred_instances') else 'N/A'}")

pred = result.pred_instances
print(f"\n=== pred_instances ===")
print(f"  bboxes shape: {pred.bboxes.shape}")
print(f"  scores shape: {pred.scores.shape}")
print(f"  labels shape: {pred.labels.shape}")
print(f"  scores 범위: {pred.scores.min().item():.4f} ~ {pred.scores.max().item():.4f}")

# conf >= 0.01 박스 수
conf_mask = pred.scores >= 0.01
print(f"\n  conf >= 0.01 박스: {conf_mask.sum().item()}")
print(f"  conf >= 0.25 박스: {(pred.scores >= 0.25).sum().item()}")
print(f"  conf >= 0.5  박스: {(pred.scores >= 0.5).sum().item()}")

# 상위 5개 박스 미리보기
top5 = pred.scores.argsort(descending=True)[:5]
print(f"\n=== 상위 5개 박스 ===")
for i, idx in enumerate(top5):
    cls = pred.labels[idx].item()
    score = pred.scores[idx].item()
    bbox = pred.bboxes[idx].tolist()
    print(f"  [{i}] class={cls} ({class_idx_to_name.get(cls, '?')[:20]}), "
          f"score={score:.3f}, bbox={[round(b,1) for b in bbox]}")

Loads checkpoint by local backend from path: /content/rtmdet_work/best_coco_bbox_mAP_epoch_15.pth
모델 로드 OK
num_classes: ('마그밀정(수산화마그네슘)', '게보린정 300mg/PTP', '알마겔정(알마게이트)(수출명:유한가스트라겔정)')... (총 118)

샘플: 게보린정 300mg_3
GT bbox 수: 1
GT class_idx: [1]

=== inference 결과 타입 ===
  type: DetDataSample
  pred_instances 키: ['scores', 'bboxes', 'labels']

=== pred_instances ===
  bboxes shape: torch.Size([300, 4])
  scores shape: torch.Size([300])
  labels shape: torch.Size([300])
  scores 범위: 0.0198 ~ 0.8369

  conf >= 0.01 박스: 300
  conf >= 0.25 박스: 1
  conf >= 0.5  박스: 1

=== 상위 5개 박스 ===
  [0] class=1 (게보린정 300mg/PTP), score=0.837, bbox=[44.6, 116.3, 215.1, 276.1]
  [1] class=23 (넥시움정 40mg), score=0.242, bbox=[44.4, 116.1, 215.2, 275.9]
  [2] class=101 (제미메트서방정 50/1000mg), score=0.235, bbox=[44.2, 115.7, 216.2, 275.5]
  [3] class=93 (아질렉트정(라사길린메실산염)), score=0.182, bbox=[45.4, 116.7, 214.7, 275.0]
  [4] class=53 (마도파정), score=0.134, bbox=[43.7, 116.2, 216.0, 275.0]


In [ ]:
import numpy as np

def iou_xyxy(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-9)

def group_and_top3(bboxes, scores, labels, iou_thr=0.5, conf_min=0.01):
    """
    raw 300 prediction → IoU 그룹핑 → 그룹별 클래스별 best conf → Top-3
    Returns: list of dict per 알약
    """
    bboxes = np.asarray(bboxes)
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    # 1. conf 필터
    mask = scores >= conf_min
    bboxes, scores, labels = bboxes[mask], scores[mask], labels[mask]
    if len(bboxes) == 0:
        return []

    # 2. conf 내림차순
    order = np.argsort(-scores)
    bboxes, scores, labels = bboxes[order], scores[order], labels[order]

    # 3. greedy 그룹핑 (시드 박스 기준 IoU>=thr 묶음)
    groups = []
    used = np.zeros(len(bboxes), dtype=bool)
    for i in range(len(bboxes)):
        if used[i]:
            continue
        members = [i]
        used[i] = True
        for j in range(i+1, len(bboxes)):
            if used[j]:
                continue
            if iou_xyxy(bboxes[i], bboxes[j]) >= iou_thr:
                members.append(j)
                used[j] = True
        groups.append(members)

    # 4. 그룹별 클래스별 best → Top-3
    results = []
    for group in groups:
        cls_best = {}  # cls -> (score, bbox)
        for idx in group:
            cls = int(labels[idx])
            if cls not in cls_best or scores[idx] > cls_best[cls][0]:
                cls_best[cls] = (float(scores[idx]), bboxes[idx])
        top3 = sorted(cls_best.items(), key=lambda kv: kv[1][0], reverse=True)[:3]
        repr_bbox = top3[0][1][1].tolist()
        results.append({
            'bbox': repr_bbox,
            'top3_class_idx': [c for c, _ in top3],
            'top3_class_name': [class_idx_to_name[c] for c, _ in top3],
            'top3_confidence': [round(v[0], 4) for _, v in top3],
        })
    return results

# ─────────────────────────────────────────
# 검증: 단일 알약 1장 + 다중 알약 1장
# ─────────────────────────────────────────
test_cases = [
    ('게보린정 300mg_3', 1),    # 단일 → 그룹 1개 기대
    ('디카테오정', 3),            # 3알 → 그룹 3개 기대
]

for stem, expected_groups in test_cases:
    if stem not in GT_LABELS:
        print(f"⚠️ {stem} GT 없음")
        continue
    img_path = GT_LABELS[stem]['img_path']
    result = inference_detector(model, img_path)
    pred = result.pred_instances

    groups = group_and_top3(
        pred.bboxes.cpu().numpy(),
        pred.scores.cpu().numpy(),
        pred.labels.cpu().numpy(),
    )

    print(f"\n=== {stem} (GT {expected_groups}알) ===")
    print(f"그룹 수: {len(groups)} (기대 {expected_groups})")
    for i, g in enumerate(groups[:3]):
        print(f"  [{i}] bbox={[round(b,1) for b in g['bbox']]}")
        for j, (name, conf) in enumerate(zip(g['top3_class_name'], g['top3_confidence'])):
            print(f"      Top-{j+1}: {name[:25]} ({conf:.3f})")


=== 게보린정 300mg_3 (GT 1알) ===
그룹 수: 32 (기대 1)
  [0] bbox=[44.6, 116.3, 215.1, 276.1]
      Top-1: 게보린정 300mg/PTP (0.837)
      Top-2: 넥시움정 40mg (0.242)
      Top-3: 제미메트서방정 50/1000mg (0.235)
  [1] bbox=[241.3, 28.9, 324.4, 327.8]
      Top-1: 타이레놀이알서방정(아세트아미노펜)(수출용) (0.067)
      Top-2: 리렉스펜정 300mg/PTP (0.037)
      Top-3: 뉴로메드정(옥시라세탐) (0.026)
  [2] bbox=[14.3, 0.0, 246.8, 16.9]
      Top-1: 락토엔큐캡슐(바실루스리케니포르미스균)  (0.058)
      Top-2: 알드린정 (0.044)
      Top-3: 람노스캡슐500mg/병 (0.039)

=== 디카테오정 (GT 3알) ===
그룹 수: 26 (기대 3)
  [0] bbox=[125.9, 150.1, 239.8, 267.3]
      Top-1: 디카테오엘씨정 10mg/병 (0.664)
      Top-2: 노바스크정 5mg (0.178)
      Top-3: 제라타딘정(로라타딘) (0.147)
  [1] bbox=[262.7, 37.7, 358.4, 167.3]
      Top-1: 디카테오엘씨정 10mg/병 (0.602)
      Top-2: 리렉스펜정 300mg/PTP (0.368)
      Top-3: 타이레놀정500mg (0.241)
  [2] bbox=[117.7, 245.6, 256.6, 325.0]
      Top-1: 뉴로메드정(옥시라세탐) (0.405)
      Top-2: 디카테오엘씨정 10mg/병 (0.344)
      Top-3: 씨즈날정(세티리진염산염) (0.255)


In [ ]:
import numpy as np

# 테스트 케이스 3개 (단일 1 / 다중 2 / 다중 3)
test_cases = [
    ('게보린정 300mg_3', 1),
    ('마이칼디정', 2),
    ('디카테오정', 3),
]
conf_levels = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]

# 미리 추론 1번씩만 (raw 박스 캐싱)
cached = {}
for stem, _ in test_cases:
    if stem not in GT_LABELS:
        continue
    result = inference_detector(model, GT_LABELS[stem]['img_path'])
    pred = result.pred_instances
    cached[stem] = (
        pred.bboxes.cpu().numpy(),
        pred.scores.cpu().numpy(),
        pred.labels.cpu().numpy(),
    )

# 스캔
print(f"{'케이스':<25} {'GT':<4} " + ' '.join(f'{c:>6}' for c in conf_levels))
print('─' * 80)
for stem, gt_n in test_cases:
    if stem not in cached:
        continue
    bboxes, scores, labels = cached[stem]
    row = f"{stem:<25} {gt_n:<4} "
    for conf in conf_levels:
        groups = group_and_top3(bboxes, scores, labels, iou_thr=0.5, conf_min=conf)
        row += f'{len(groups):>6} '
    print(row)

# 추가: 게보린 단일에서 conf별 Top-3 정답 유지되는지
print("\n=== 게보린정 (GT=1, dl_name='게보린정 300mg/PTP') Top-1 박스 변화 ===")
bboxes, scores, labels = cached['게보린정 300mg_3']
for conf in conf_levels:
    groups = group_and_top3(bboxes, scores, labels, iou_thr=0.5, conf_min=conf)
    if groups:
        g0 = groups[0]
        top1 = g0['top3_class_name'][0]
        top1_conf = g0['top3_confidence'][0]
        n_top3 = len(g0['top3_class_name'])
        match = '✅' if top1 == '게보린정 300mg/PTP' else '❌'
        print(f"  conf={conf}: {match} Top-1='{top1[:20]}' ({top1_conf:.3f}), Top-3 후보 {n_top3}개")
    else:
        print(f"  conf={conf}: 그룹 0개")

케이스                       GT     0.01   0.05    0.1    0.2    0.3    0.5
────────────────────────────────────────────────────────────────────────────────
게보린정 300mg_3              1        32      3      1      1      1      1 
마이칼디정                     2        13     13     10      5      2      0 
디카테오정                     3        26      4      3      3      3      2 

=== 게보린정 (GT=1, dl_name='게보린정 300mg/PTP') Top-1 박스 변화 ===
  conf=0.01: ✅ Top-1='게보린정 300mg/PTP' (0.837), Top-3 후보 3개
  conf=0.05: ✅ Top-1='게보린정 300mg/PTP' (0.837), Top-3 후보 3개
  conf=0.1: ✅ Top-1='게보린정 300mg/PTP' (0.837), Top-3 후보 3개
  conf=0.2: ✅ Top-1='게보린정 300mg/PTP' (0.837), Top-3 후보 3개
  conf=0.3: ✅ Top-1='게보린정 300mg/PTP' (0.837), Top-3 후보 1개
  conf=0.5: ✅ Top-1='게보린정 300mg/PTP' (0.837), Top-3 후보 1개


In [ ]:
import json, time, os
from datetime import datetime

CONF_THR = 0.05
IOU_GROUP_THR = 0.5
IOU_MATCH_THR = 0.5

all_results = []
total_inference_ms = 0
n_top1_correct = 0
n_top3_correct = 0
n_matched = 0
total_gt = 0
img_top1_full = 0  # 이미지 단위 (모든 GT가 Top-1 정답)
img_top3_full = 0
n_false_pos = 0    # 매칭 안 된 prediction 그룹 (오탐)

for stem in sorted(GT_LABELS.keys()):
    img_path = GT_LABELS[stem]['img_path']
    gt_bboxes = GT_LABELS[stem]['bboxes']
    total_gt += len(gt_bboxes)

    t0 = time.time()
    result = inference_detector(model, img_path)
    pred = result.pred_instances
    total_inference_ms += (time.time() - t0) * 1000

    pred_groups = group_and_top3(
        pred.bboxes.cpu().numpy(),
        pred.scores.cpu().numpy(),
        pred.labels.cpu().numpy(),
        iou_thr=IOU_GROUP_THR,
        conf_min=CONF_THR,
    )

    # GT 각각에 IoU 가장 높은 prediction 매칭 (greedy)
    gt_results = []
    used_pred = set()
    img_all_top1 = True
    img_all_top3 = True

    for gt in gt_bboxes:
        best_iou, best_idx = 0.0, -1
        for j, pg in enumerate(pred_groups):
            if j in used_pred:
                continue
            iou = iou_xyxy(gt['bbox'], pg['bbox'])
            if iou > best_iou:
                best_iou, best_idx = iou, j

        gr = {
            'gt_dl_name': gt['dl_name'],
            'gt_bbox': [round(v, 1) for v in gt['bbox']],
            'matched_iou': round(best_iou, 4),
            'top1_correct': False,
            'top3_correct': False,
        }
        if best_iou >= IOU_MATCH_THR and best_idx >= 0:
            n_matched += 1
            used_pred.add(best_idx)
            pg = pred_groups[best_idx]
            gr['matched_top3_class_name'] = pg['top3_class_name']
            gr['matched_top3_confidence'] = pg['top3_confidence']
            if pg['top3_class_name'][0] == gt['dl_name']:
                gr['top1_correct'] = True
                n_top1_correct += 1
            else:
                img_all_top1 = False
            if gt['dl_name'] in pg['top3_class_name']:
                gr['top3_correct'] = True
                n_top3_correct += 1
            else:
                img_all_top3 = False
        else:
            img_all_top1 = False
            img_all_top3 = False
        gt_results.append(gr)

    if img_all_top1: img_top1_full += 1
    if img_all_top3: img_top3_full += 1
    n_false_pos += len(pred_groups) - len(used_pred)

    all_results.append({
        'filename': os.path.basename(img_path),
        'gt_bboxes': [{'bbox': [round(v,1) for v in b['bbox']], 'dl_name': b['dl_name']} for b in gt_bboxes],
        'predictions': pred_groups,
        'gt_results': gt_results,
    })

top1_acc = n_top1_correct / total_gt
top3_acc = n_top3_correct / total_gt
detect_rate = n_matched / total_gt
ms_per_img = total_inference_ms / 33

print(f"=== RTMDet-tiny 채점 결과 (conf={CONF_THR}) ===")
print(f"GT bbox: {total_gt}")
print(f"Detection (IoU≥0.5): {n_matched}/{total_gt} ({detect_rate:.1%})")
print(f"Top-1 정답: {n_top1_correct}/{total_gt} ({top1_acc:.1%})")
print(f"Top-3 정답: {n_top3_correct}/{total_gt} ({top3_acc:.1%})")
print(f"이미지 단위 Top-1 전부 정답: {img_top1_full}/33 ({img_top1_full/33:.1%})")
print(f"이미지 단위 Top-3 전부 정답: {img_top3_full}/33 ({img_top3_full/33:.1%})")
print(f"오탐 그룹 수: {n_false_pos}")
print(f"ms/img: {ms_per_img:.1f}")
print()
print(f"=== 임계값 판정 ===")
print(f"베이스라인 Top-3:  25.5% (12/47)")
print(f"RTMDet-tiny Top-3: {top3_acc:.1%} ({n_top3_correct}/47)")
print(f"통과: {'✅' if top3_acc >= 0.255 else '❌'}")

output = {
    'exp_name': 'rtmdet_tiny_synth_e15_seed100',
    'author': '승원',
    'model_framework': 'mmdet rtmdet-tiny',
    'model_params_M': 4.91,
    'batch_size_used': 16,
    'seed': 100,
    'conf_threshold': CONF_THR,
    'nms_iou': IOU_GROUP_THR,
    'iou_match_threshold': IOU_MATCH_THR,
    'eval_version': 'v4',
    'num_images': 33,
    'num_gt_bboxes': total_gt,
    'metrics': {
        'detection_rate': round(detect_rate, 4),
        'top1_accuracy': round(top1_acc, 4),
        'top3_accuracy': round(top3_acc, 4),
        'image_top1_full_rate': round(img_top1_full/33, 4),
        'image_top3_full_rate': round(img_top3_full/33, 4),
        'false_positive_groups': n_false_pos,
        'ms_per_image': round(ms_per_img, 2),
    },
    'evaluated_at': datetime.now().isoformat(),
    'results': all_results,
}
with open('/content/rtmdet_tiny_synth_e15_seed100_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f"\n저장: /content/rtmdet_tiny_synth_e15_seed100_results.json")

=== RTMDet-tiny 채점 결과 (conf=0.05) ===
GT bbox: 47
Detection (IoU≥0.5): 45/47 (95.7%)
Top-1 정답: 14/47 (29.8%)
Top-3 정답: 18/47 (38.3%)
이미지 단위 Top-1 전부 정답: 9/33 (27.3%)
이미지 단위 Top-3 전부 정답: 11/33 (33.3%)
오탐 그룹 수: 265
ms/img: 117.9

=== 임계값 판정 ===
베이스라인 Top-3:  25.5% (12/47)
RTMDet-tiny Top-3: 38.3% (18/47)
통과: ✅

저장: /content/rtmdet_tiny_synth_e15_seed100_results.json


In [ ]:
import shutil, os, glob
from google.colab import files

# 백업 폴더 정리
backup_dir = '/content/rtmdet_backup'
shutil.rmtree(backup_dir, ignore_errors=True)
os.makedirs(backup_dir, exist_ok=True)

# 박제 대상
backup_items = [
    '/content/rtmdet_work/best_coco_bbox_mAP_epoch_15.pth',
    '/content/rtmdet_work/rtmdet_tiny_pill.py',
    '/content/rtmdet_tiny_synth_e15_seed100_results.json',
    '/content/train_coco.json',
    '/content/val_coco.json',
]

for src in backup_items:
    if os.path.exists(src):
        shutil.copy(src, backup_dir)
        size_mb = os.path.getsize(src) / 1024 / 1024
        print(f"  ✅ {os.path.basename(src)} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {src} 없음")

# 학습 로그도 챙김 (work-dir 안)
log_dirs = sorted(glob.glob('/content/rtmdet_work/2026*'))
if log_dirs:
    latest_log = log_dirs[-1]
    log_backup = os.path.join(backup_dir, os.path.basename(latest_log))
    shutil.copytree(latest_log, log_backup)
    log_size = sum(os.path.getsize(os.path.join(r, f))
                   for r, _, fs in os.walk(log_backup) for f in fs) / 1024 / 1024
    print(f"  ✅ 학습 로그 폴더 {os.path.basename(latest_log)} ({log_size:.1f} MB)")

# zip
zip_path = '/content/rtmdet_tiny_backup'
shutil.make_archive(zip_path, 'zip', backup_dir)
final = zip_path + '.zip'
print(f"\n📦 zip 생성: {final} ({os.path.getsize(final)/1024/1024:.1f} MB)")

# 다운로드 트리거
files.download(final)

  ✅ best_coco_bbox_mAP_epoch_15.pth (41.7 MB)
  ✅ rtmdet_tiny_pill.py (0.0 MB)
  ✅ rtmdet_tiny_synth_e15_seed100_results.json (0.2 MB)
  ✅ train_coco.json (6.8 MB)
  ✅ val_coco.json (1.6 MB)
  ✅ 학습 로그 폴더 20260429_060231 (0.3 MB)

📦 zip 생성: /content/rtmdet_tiny_backup.zip (39.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Colab 셀 1: 학습 yaml 확인
!cat /content/dataset_yolo/data.yaml

cat: /content/dataset_yolo/data.yaml: No such file or directory


In [ ]:
# Colab 셀 2: stem 리스트 추출
import os

train_dir = '/content/dataset_yolo/images/train'  # yaml의 train 경로
val_dir = '/content/dataset_yolo/images/val'

train_stems = sorted([os.path.splitext(f)[0] for f in os.listdir(train_dir)])
val_stems = sorted([os.path.splitext(f)[0] for f in os.listdir(val_dir)])

print(f"train: {len(train_stems)}장")
print(f"val:   {len(val_stems)}장")

# Drive에 저장
import os
os.makedirs('/content/drive/MyDrive/pill_detection/eval', exist_ok=True)
with open('/content/drive/MyDrive/pill_detection/eval/train_stems.txt', 'w') as f:
    f.write('\n'.join(train_stems))
with open('/content/drive/MyDrive/pill_detection/eval/val_stems.txt', 'w') as f:
    f.write('\n'.join(val_stems))

print("Drive 저장 완료")

train: 8302장
val:   2090장
Drive 저장 완료
